# 🚀 BigQuery DML 연동 Apache Iceberg 테이블 생성 및 검증 테스트 (`bq_iceberg_bigquery_dml.ipynb`)

본 주피터 노트북은 BigLake REST Catalog 기반 **Lakehouse Iceberg Table** 생성 시 `'gcp.biglake.bigquery-dml.enabled' = 'true'` 프로퍼티 및 `DayTransform()` 파티셔닝을 적용하여, BigQuery 엔진에서 직접 DML(INSERT, UPDATE, DELETE) 수행이 가능하도록 테이블을 프로비저닝하고 검증합니다.

### 💡 주요 특징:
1. **간결한 단일 목적 환경**: 벤치마킹 및 타 테이블 아키텍처 코드를 제외하고 BigQuery DML 연동 테스트 전용으로 구성.
2. **BigLake Iceberg REST Catalog 연동**: PyIceberg 기반 schema, partition spec (`DayTransform`), table properties 정의.
3. **`gcp.biglake.bigquery-dml.enabled` = true 옵션 내장**.
4. **BigQuery DML (INSERT / UPDATE / DELETE) 동작 실시간 검증 기능 제공**.

## [1단계: 환경 설정, JVM 모듈 개방 및 GCP 리소스 설정]

작업에 필요한 패키지를 로드하고 JVM 개방 플래그 및 GCP Project, Dataset, Connection 식별자를 정의합니다.

In [ ]:
# =========================================================================
# ⚙️ [1단계: JVM 모듈 개방, 라이브러리 로드 및 GCP 변수 설정]
# =========================================================================
import os
import time
import uuid
import subprocess
import google.auth
import pandas as pd
import numpy as np
import pyarrow as pa

from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound, Conflict
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, TimestampType, DateType, DoubleType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform

# 💡 JDK 17+/26+ 호환성을 위한 JVM 모듈 개방 플래그 주입
os.environ["JDK_JAVA_OPTIONS"] = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED"
)

# =========================================================================
# ⚙️ [GCP 프로젝트 및 리소스 식별자 설정]
# =========================================================================
USER_PROJECT_ID = None  # 특정 GCP Project ID 지정 시 문자열 입력 (예: "my-gcp-project-id")

PROJECT_ID = USER_PROJECT_ID or os.getenv("GCP_PROJECT_ID")

try:
    credentials, auth_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    if not PROJECT_ID and auth_project:
        PROJECT_ID = auth_project
except Exception:
    credentials = None

if not PROJECT_ID:
    PROJECT_ID = "your-gcp-project-id"

DATASET_NAME = "bq_iceberg_benchmark_ds"
CONNECTION_NAME = "lakehouse-iceberg-conn"
BUCKET_NAME = f"{PROJECT_ID}-bq-iceberg-demo-bucket"

# Dataset 리전 자동 감지 (기본: asia-northeast3 서울)
temp_bq = bigquery.Client(project=PROJECT_ID, credentials=credentials)
REGION = os.getenv("GCP_REGION", "asia-northeast3")
try:
    ds_obj = temp_bq.get_dataset(f"{PROJECT_ID}.{DATASET_NAME}")
    REGION = ds_obj.location
except Exception:
    pass

# BQ & GCS 클라이언트 초기화
bq_client = bigquery.Client(project=PROJECT_ID, location=REGION, credentials=credentials)
gcs_client = storage.Client(project=PROJECT_ID, credentials=credentials)

print("=========================================================")
print(f"🎯 설정된 GCP Project ID : {PROJECT_ID}")
print(f"🎯 설정된 GCP Region     : {REGION}")
print(f"🎯 설정된 GCS 버킷 이름  : {BUCKET_NAME}")
print(f"🎯 설정된 BigQuery Dataset : {DATASET_NAME}")
print("=========================================================")

## [2단계: GCP Lakehouse & BigQuery 완전 자동화 인프라 프로비저닝]

GCS 버킷, BigLake Catalog 자원, BigQuery Connection, BigQuery Dataset 및 IAM 객체 관리 권한(`roles/storage.objectAdmin`)을 확인 및 프로비저닝합니다.

In [ ]:
# =========================================================================
# ⚙️ [2단계: GCS 버킷, BigLake Catalog, Connection, Dataset 자동 생성]
# =========================================================================

# 1. GCS 버킷 생성/바인딩
try:
    existing_b = gcs_client.get_bucket(BUCKET_NAME)
    bucket = existing_b
    print(f"ℹ️ GCS 버킷 바인딩 완료: gs://{BUCKET_NAME}")
except Exception:
    bucket = gcs_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"✅ GCS 버킷 생성 완료: gs://{BUCKET_NAME}")

# 2. BigLake Catalog 생성 (gcloud CLI)
CATALOG_ID = BUCKET_NAME
try:
    print(f"🔄 BigLake Iceberg Catalog ({CATALOG_ID}) 프로비저닝 중...")
    create_cmd = f"gcloud biglake iceberg catalogs create {CATALOG_ID} --project={PROJECT_ID} --catalog-type=gcs_bucket"
    cat_res = subprocess.run(create_cmd, shell=True, capture_output=True, text=True)
    out_msg = cat_res.stdout.strip() or cat_res.stderr.strip() or "OK"
    print(f"✅ BigLake Iceberg Catalog 상태: {out_msg}")
except Exception as e:
    print(f"ℹ️ Catalog 안내: {e}")

# 3. BigQuery Connection & IAM 권한 부여
sa_email = None
bq_sa_email = None

try:
    from google.cloud import bigquery_connection_v1 as bq_connection
    conn_client = bq_connection.ConnectionServiceClient(credentials=credentials)
    location_path = f"projects/{PROJECT_ID}/locations/{REGION}"
    conn_name_path = f"{location_path}/connections/{CONNECTION_NAME}"
    
    try:
        conn = conn_client.get_connection(request={"name": conn_name_path})
        sa_email = conn.cloud_resource.service_account_id
        print(f"✅ BigQuery Connection 존재함: {CONNECTION_NAME} (SA: {sa_email})")
    except Exception:
        print(f"🔄 BigQuery Connection ({CONNECTION_NAME}) 생성 중...")
        conn_obj = bq_connection.Connection(cloud_resource=bq_connection.CloudResourceProperties())
        conn = conn_client.create_connection(
            request={"parent": location_path, "connection_id": CONNECTION_NAME, "connection": conn_obj}
        )
        sa_email = conn.cloud_resource.service_account_id
        print(f"✅ BigQuery Connection 생성 완료: {CONNECTION_NAME} (SA: {sa_email})")
except Exception as e:
    print(f"ℹ️ Connection 생성 안내: {e}")

try:
    bq_sa_email = bq_client.get_service_account_email()
    print(f"🔑 BigQuery Service Agent 이메일: {bq_sa_email}")
except Exception:
    pass

# IAM 권한 자동 부여 (Storage Object Admin)
try:
    policy = bucket.get_iam_policy(requested_policy_version=3)
    members_to_add = set()
    if sa_email: members_to_add.add(f"serviceAccount:{sa_email}")
    if bq_sa_email: members_to_add.add(f"serviceAccount:{bq_sa_email}")
    if members_to_add:
        policy.bindings.append({"role": "roles/storage.objectAdmin", "members": list(members_to_add)})
        bucket.set_iam_policy(policy)
        print(f"✅ Connection SA 및 BigQuery SA 계정에 GCS Storage Object Admin IAM 권한 부여 완료!")
except Exception as e:
    print(f"⚠️ IAM 설정 결과: {e}")

# 4. BigQuery Dataset 생성
dataset_id = f"{PROJECT_ID}.{DATASET_NAME}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = REGION
try:
    dataset = bq_client.create_dataset(dataset, timeout=30)
    print(f"✅ BigQuery Dataset 생성 완료: {dataset_id}")
except Conflict:
    print(f"ℹ️ BigQuery Dataset 이미 존재함: {dataset_id}")

## [3단계: BigQuery DML 연동 Lakehouse Iceberg Table 생성]

`DayTransform()` 파티션 및 `'gcp.biglake.bigquery-dml.enabled' = 'true'` 프로퍼티를 설정하여 PyIceberg로 REST Catalog 테이블을 생성하고 기본 초깃값 데이터를 적재합니다.

In [ ]:
# =========================================================================
# ⚙️ [3단계: BigQuery DML 옵션 적용 Lakehouse Iceberg Table 생성]
# =========================================================================
lakehouse_iceberg_id = f"{PROJECT_ID}.{BUCKET_NAME}.default.external_dml_weblog"

if os.path.exists("/tmp/pyiceberg_catalog.db"):
    try: os.remove("/tmp/pyiceberg_catalog.db")
    except Exception: pass

# GCS Warehouse 사전 생성
try:
    wh_blob = bucket.blob("external_iceberg_warehouse/.keep")
    if not wh_blob.exists():
        wh_blob.upload_from_string("keep", content_type="text/plain")
except Exception:
    pass

BIGLAKE_REST_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"
GCS_WAREHOUSE_URI = f"gs://{BUCKET_NAME}/external_iceberg_warehouse"
LAKEHOUSE_BL_WAREHOUSE = f"bl://projects/{PROJECT_ID}/catalogs/{CATALOG_ID}"
token_str = getattr(credentials, 'token', '') or ""

print(f"🔄 GCP BigLake REST Catalog 연동 및 PyIceberg 테이블 생성 시작...")
print(f"  - Lakehouse Iceberg Table ID: {lakehouse_iceberg_id}")
print(f"  - BigLake REST Catalog URI  : {BIGLAKE_REST_URI}")
print(f"  - GCS Storage Warehouse     : {GCS_WAREHOUSE_URI}")

try:
    rest_options = {
        "type": "rest",
        "uri": BIGLAKE_REST_URI,
        "warehouse": GCS_WAREHOUSE_URI,
        "header.x-goog-user-project": PROJECT_ID,
        "header.X-Iceberg-Access-Delegation": "vended-credentials"
    }
    if token_str: rest_options["token"] = token_str
    
    try:
        pyiceberg_cat = load_catalog("lakehouse_catalog", **rest_options)
    except Exception:
        rest_options["warehouse"] = LAKEHOUSE_BL_WAREHOUSE
        pyiceberg_cat = load_catalog("lakehouse_catalog", **rest_options)
    
    schema = Schema(
        NestedField(field_id=1, name="event_id", field_type=StringType(), required=False),
        NestedField(field_id=2, name="event_timestamp", field_type=TimestampType(), required=False),
        NestedField(field_id=3, name="event_date", field_type=DateType(), required=False),
        NestedField(field_id=4, name="user_id", field_type=StringType(), required=False),
        NestedField(field_id=5, name="event_type", field_type=StringType(), required=False),
        NestedField(field_id=6, name="amount", field_type=DoubleType(), required=False),
        NestedField(field_id=7, name="device_os", field_type=StringType(), required=False),
    )
    
    partition_spec = PartitionSpec(
        PartitionField(source_id=3, field_id=1000, transform=DayTransform(), name="event_date_day")
    )
    
    try: pyiceberg_cat.create_namespace("default")
    except Exception: pass

    try: pyiceberg_cat.drop_table("default.external_dml_weblog")
    except Exception: pass

    # Sort Order 프로퍼티 세팅 및 BigQuery DML 옵션 추가
    iceberg_ext_tbl = pyiceberg_cat.create_table(
        identifier="default.external_dml_weblog",
        schema=schema,
        partition_spec=partition_spec,
        properties={
            "write.sort.order": "user_id ASC NULLS FIRST, event_type ASC NULLS FIRST",
            "gcp.biglake.bigquery-dml.enabled": "true"
        }
    )
    
    # 초깃값 더미 테스트 데이터 생성 (1,000건)
    dates = pd.date_range(start="2026-07-01", end="2026-07-05", freq="D")
    event_types = ['CLICK', 'PURCHASE', 'VIEW', 'LOGOUT']
    device_oses = ['iOS', 'Android', 'Web']
    
    sample_df = pd.DataFrame({
        'event_id': [str(uuid.uuid4()) for _ in range(1000)],
        'event_timestamp': pd.date_range(start="2026-07-01", periods=1000, freq="1s"),
        'event_date': np.random.choice(dates.date, size=1000),
        'user_id': [f"USER_{np.random.randint(1000, 9999)}" for _ in range(1000)],
        'event_type': np.random.choice(event_types, size=1000),
        'amount': np.round(np.random.exponential(scale=50.0, size=1000), 2),
        'device_os': np.random.choice(device_oses, size=1000)
    })
    
    pa_table = pa.Table.from_pandas(sample_df, preserve_index=False)
    iceberg_ext_tbl.append(pa_table)
    
    print(f"✅ Lakehouse Iceberg Table (`gcp.biglake.bigquery-dml.enabled` = true) 생성 및 샘플 데이터 적재 완료: {lakehouse_iceberg_id}")
except Exception as e:
    print(f"⚠️ Lakehouse Iceberg Table 생성 결과: {e}")

## [4단계: BigQuery SQL DML (UPDATE / INSERT / DELETE) 동작 검증]

생성된 BigLake Iceberg Table(`f"{PROJECT_ID}.{BUCKET_NAME}.default.external_dml_weblog"`)에 대해 BigQuery 엔진에서 DML 문이 정상 동작하는지 테스트합니다.

In [ ]:
# =========================================================================
# ⚙️ [4단계: BigQuery DML 쿼리 실행 검증]
# =========================================================================
target_table = f"{PROJECT_ID}.{BUCKET_NAME}.default.external_dml_weblog"

print(f"🧪 [DML 검증 1] SELECT COUNT(*) 조회...")
res = list(bq_client.query(f"SELECT COUNT(*) as cnt FROM `{target_table}`").result())
print(f"  - 초기 행 수: {res[0]['cnt']} 건")

print(f"\n🧪 [DML 검증 2] INSERT INTO 테스트...")
insert_sql = f"""
INSERT INTO `{target_table}` (event_id, event_timestamp, event_date, user_id, event_type, amount, device_os)
VALUES ('DML_TEST_001', CURRENT_DATETIME(), CURRENT_DATE(), 'USER_TEST', 'PURCHASE', 999.99, 'Web');
"""
try:
    bq_client.query(insert_sql).result()
    print(f"  ✅ INSERT 성공!")
except Exception as e:
    print(f"  ❌ INSERT 실패: {e}")

print(f"\n🧪 [DML 검증 3] UPDATE 테스트...")
update_sql = f"""
UPDATE `{target_table}`
SET amount = 888.88
WHERE event_id = 'DML_TEST_001';
"""
try:
    bq_client.query(update_sql).result()
    print(f"  ✅ UPDATE 성공!")
except Exception as e:
    print(f"  ❌ UPDATE 실패: {e}")

print(f"\n🧪 [DML 검증 4] DELETE 테스트...")
delete_sql = f"""
DELETE FROM `{target_table}`
WHERE event_id = 'DML_TEST_001';
"""
try:
    bq_client.query(delete_sql).result()
    print(f"  ✅ DELETE 성공!")
except Exception as e:
    print(f"  ❌ DELETE 실패: {e}")